# Data Loading & Reformatting

In this notebook, we'll download the Metacritic data set from Kaggle and check whether everything is represented correctly.  
We will also reformat the data frame. The original data contains ~300 variables that are a flattened representation of the system-dependent review counts from the corresponding game. This should be reformatted to be made usable.

## Setup

In [1]:
# +++ Import necessary modules +++

import re

import numpy as np
import pandas as pd
from IPython.display import display

from data.download import load_from_kaggle
from eda.overview import overview


In [2]:
# +++ Use shared project paths +++
from core.config import (
    DATA_FORMATTED_PATH,
    DATA_RAW_PATH,
    PROCESSED_DATA_DIR,
    RAW_DATA_DIR,
)


## Load and save data set

In [3]:
# +++ Get data set from kaggle website and save to raw folder +++

full_link = r"https://www.kaggle.com/datasets/zaireali/metacritic-games-scrape"
dataset_link = full_link.split("/datasets/")[-1]

destination = str(RAW_DATA_DIR)
dataset_name = dataset_link.split('/')[1]

print(f'📦 Loading data set: {dataset_name}')
files = load_from_kaggle(dataset_link = dataset_link,
                         destination = destination,
                         create_subfolder = False)

print(f'✅ {len(files)} File(s) found:')
for i, file in enumerate(files, 1):
    print(f"   {i}. {file}")

📦 Loading data set: metacritic-games-scrape
Destination directory 'C:\Users\janos\Projects\StackFuel_PP\data\raw' already exists with files. Skipping download (replace=False).
✅ 2 File(s) found:
   1. .gitkeep
   2. dataset_metacritic_scraper_2025-02-15.csv


## Read data and do first inspection

In [4]:
# +++ Load data into workspace +++

df = pd.read_csv(DATA_RAW_PATH)

<positron-console-cell-4>:3: DtypeWarning: Columns (2,155,162,163,170,171,178,179,186,195,196,197,198) have mixed types. Specify dtype option on import or set low_memory=False.


In [5]:
# some meta information about the data set

print(f"\n🔢 Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"🔄 Duplicates: {df.duplicated().sum():,} ({df.duplicated().sum()/len(df)*100:.2f}%)")
print(f"💾 Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")


🔢 Shape: 13,429 rows × 308 columns
🔄 Duplicates: 0 (0.00%)
💾 Memory Usage: 83.98 MB


We already know that all variables after 'userscore' are the flattened output of the scraper that needs to be reformatted.  
So for now, we focus on the first 11 variables in our initial check

In [6]:
# +++ Check info on first 11 variables +++

main_vars = df.columns[:11]

df[main_vars].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13429 entries, 0 to 13428
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   title          13429 non-null  object
 1   genres/0       13429 non-null  object
 2   metascore      13429 non-null  object
 3   publisherName  13427 non-null  object
 4   publisherUrl   13427 non-null  object
 5   releaseDate    13397 non-null  object
 6   section        13429 non-null  object
 7   summary        13385 non-null  object
 8   type           13429 non-null  object
 9   url            13429 non-null  object
 10  userscore      13429 non-null  object
dtypes: object(11)
memory usage: 1.1+ MB


In [7]:
# +++ tackle the Dtypewarning from reading in the data

mixed_cols = [2, 155, 162, 163, 170, 171, 178, 179, 186, 195, 196, 197, 198]

display(df.iloc[:, mixed_cols].dtypes)

for col in df.columns[mixed_cols]:
    print(col)
    print(df[col].map(type).value_counts())

metascore                  object
platformReviews/8/name     object
platformReviews/8/url      object
platformReviews/9/name     object
platformReviews/9/url      object
platformReviews/10/name    object
platformReviews/10/url     object
platformReviews/11/name    object
platformReviews/11/url     object
platforms/8                object
platforms/9                object
platforms/10               object
platforms/11               object
dtype: object

metascore
metascore
<class 'str'>    11381
<class 'int'>     2048
Name: count, dtype: int64
platformReviews/8/name
platformReviews/8/name
<class 'float'>    13392
<class 'str'>         37
Name: count, dtype: int64
platformReviews/8/url
platformReviews/8/url
<class 'float'>    13392
<class 'str'>         37
Name: count, dtype: int64
platformReviews/9/name
platformReviews/9/name
<class 'float'>    13412
<class 'str'>         17
Name: count, dtype: int64
platformReviews/9/url
platformReviews/9/url
<class 'float'>    13412
<class 'str'>         17
Name: count, dtype: int64
platformReviews/10/name
platformReviews/10/name
<class 'float'>    13426
<class 'str'>          3
Name: count, dtype: int64
platformReviews/10/url
platformReviews/10/url
<class 'float'>    13426
<class 'str'>          3
Name: count, dtype: int64
platformReviews/11/name
platformReviews/11/name
<class 'float'>    13428
<class 'str'>          1
Name: count, dtype: int64
platformReviews/11/url
platformReviews/11/url
<class '

It appears the we have variables that should be numerical, but are of dtype object because there are some string numbers mixed in.

In [8]:
# convert metascore to integer
df['metascore'] = pd.to_numeric(df['metascore'], errors = 'coerce').astype('Int32')

# convert userscore to float
df['userscore'] = pd.to_numeric(df['userscore'], errors = 'coerce').astype('float32')

# convert the rest of flagged variables to string
for col in df.columns[mixed_cols[1:]]:
    df[col] = df[col].astype('string')

# convert the release date to datatime format
df['releaseDate'] = pd.to_datetime(df['releaseDate'])

In [9]:
# Use overview function on the "main" vars
ov, num_vars, cat_vars = overview(df[main_vars])

Duplicates: 0

Auto Sales Data - Variable Overview


,dtype,total,missing_n,missing_%,uniques_n,uniques
title,object,13429,0,0.000000,13429,"[Tekken 3, Mass Effect 2, Baldur's Gate 3, The..."
genres/0,object,13429,0,0.000000,119,"[3D Fighting, Western RPG, Compilation, Linear..."
metascore,Int32,13422,7,0.052126,84,"[96, 97, 98, 99, 95, 94, 93, 92, 91, 90, 89, <..."
publisherName,object,13427,2,0.014893,2038,"[Namco, Electronic Arts, Larian Studios Games,..."
publisherUrl,object,13427,2,0.014893,2038,"[https://www.metacritic.com/company/namco/, ht..."
releaseDate,datetime64[ns],13397,32,0.238290,4761,"[1998-04-29 00:00:00, 2010-01-26 00:00:00, 202..."
section,object,13429,0,0.000000,22,"[PlayStation, Xbox 360, PC, PlayStation 3, Gam..."
summary,object,13385,44,0.327649,13314,"[An ancient evil force has reawakened, attacki..."
type,object,13429,0,0.000000,1,[game]
url,object,13429,0,0.000000,13429,"[https://www.metacritic.com/game/tekken-3, htt..."



Descriptive Metrics on numeric variables


,metascore,releaseDate,userscore
count,13422.0,13397,11896.000000
mean,70.407018,2012-08-22 01:41:08.701948160,6.929203
min,11.0,1995-04-30 00:00:00,0.300000
25%,63.0,2006-10-31 00:00:00,6.300000
50%,72.0,2012-08-08 00:00:00,7.200000
75%,79.0,2018-08-03 00:00:00,7.900000
max,99.0,2025-02-28 00:00:00,10.000000
std,12.350829,NaN,1.358217



Numeric Variables in the data set


,dtype
metascore,Int32
userscore,float32



Non-numeric variables in the data set


,dtype,n_uniques
title,object,13429
genres/0,object,119
publisherName,object,2038
publisherUrl,object,2038
section,object,22
summary,object,13314
type,object,1
url,object,13429


**Insights**  
- Meta data contains mostly categorical or text information
- main numerical variables: metascore, userscore
- Type is useless, as it contains "game" in every entry -> remove
- genres and publisherName have A LOT of categories. Need way to deal with that

## Which Info is actually represented in the flattened variable space?

To extract information, we'll need to be certain about what kind of information is represented in the `[platform]UserReviews/..`, `platformReviews/<index>/..`, and `userReviewsSummary/..` variables.  
  
- Critic score variables range from 0 to 100
- User score variables should from from 0.0 to 10.0

In [10]:
# Check score variables for range

score_cols = df.columns[df.columns.str.contains('score', case = True, regex = False)]

scores_num = df[score_cols].apply(pd.to_numeric, errors = 'coerce')

scores_num.describe().transpose()

,count,mean,std,min,25%,50%,75%,max
metascore,13422.0,70.407018,12.350829,11.0,63.0,72.0,79.0,99.0
userscore,11896.0,6.929203,1.358217,0.3,6.3,7.2,7.9,10.0
3DsUserReviews/score,426.0,69.157277,12.503408,23.0,62.0,71.0,78.0,94.0
dreamcastUserReviews/score,134.0,72.738806,14.141035,29.0,63.25,75.0,83.0,98.0
dsUserReviews/score,797.0,65.539523,13.021243,19.0,58.0,67.0,75.0,93.0
gameBoyAdvanceUserReviews/score,486.0,67.895062,13.264451,24.0,59.25,70.0,77.0,95.0
gameCubeUserReviews/score,483.0,69.761905,13.127338,23.0,62.0,70.0,80.0,97.0
iOsIPhoneIPadUserReviews/score,445.0,77.937079,10.413115,24.0,73.0,79.0,86.0,98.0
metaQuestUserReviews/score,17.0,78.823529,5.581614,68.0,76.0,80.0,84.0,86.0
nintendo64UserReviews/score,79.0,77.772152,13.304086,41.0,72.5,80.0,88.0,99.0


==> Only `userscore`and `userReviewsSummary/score` seem to have a range from 0.0 to 10.0 with decimals

All other variables range up to 100 and appear to be integers.

In [11]:
# Check variables containing URLs for the review pages

# select all variables containing urls (except game and publisher url)
url_cols = [col for col in df.columns
            if 'url' in col.lower() and col not in {'url', 'publisherUrl'}]

url_results = []
for col in url_cols:

    urls = df[col].astype('string').str.strip()
    non_missing = urls.notna() & urls.ne('')
    match_critic = urls.str.contains('critic-reviews', case = False, regex = False, na = False)
    match_user = urls.str.contains('user-reviews', case = False, regex = False, na = False)

    total = non_missing.sum()
    matching_critic = (non_missing & match_critic).sum()
    matching_user = (non_missing & match_user).sum()

    url_results.append({
        'column': col, 
        'non_missing_count': total,
        'critic_reviews_count': matching_critic,
        'user_reviews_count': matching_user, 
    })

pd.DataFrame(url_results)

,column,non_missing_count,critic_reviews_count,user_reviews_count
0,3DsUserReviews/url,498,498,0
1,dreamcastUserReviews/url,196,196,0
2,dsUserReviews/url,911,911,0
3,gameBoyAdvanceUserReviews/url,583,583,0
4,gameCubeUserReviews/url,564,564,0
5,iOsIPhoneIPadUserReviews/url,1282,1282,0
6,metaQuestUserReviews/url,112,112,0
7,nintendo64UserReviews/url,98,98,0
8,nintendoSwitchUserReviews/url,3184,3184,0
9,pcUserReviews/url,8364,8364,0


==> There is not a single url linking user reviews
==> All urls link to critic review pages

**Bottom line**
- `userReviewsSuammry/..` variables are assumed to represent user review information (score, counts)
- `[platform]UserReviews/..` and `platformReviews/<index>/..` seem to contain duplicated info on critic score and review count
- `metascore`and `userscore` are assumed to show the review scores for the platform listed in `section`

## Reformatting of the data set

**Goal**
- Harness the information from critic and user review counts
- transform the data into a game x platform format, where every row corresponds to a platform a particular game has reviews on metacritic on
    - this will create missing values, because we do not have critic and user scores available for all platforms
    - these will be filled by a web scrape later

In [12]:
# +++ Start the reformatting process +++

# --- Which info goes where? ---

# 1. Define metadata to repeat on every platform row.
#    Matches original column names (keys) to new ones for the new table format.
#    Will be repeated for every platform row of a game entry.
metadata = {
    "title": "title",
    "genres/0": "genre",
    "summary": "summary",
    "publisherName": "publisherName",
    "publisherUrl": "publisherUrl",
    "url": "url",
    "section": "original_platform",
    "releaseDate": "ReleaseDate",
}

# 2. How to extract the measures inside each "platformReviews/<index>/..." group.  
critic_fields = {
    "score": "metascore",
    "normalizedScore": "critic_normalized_score", # normalized score gets its own new column
    "positiveCount": "critic_positive_count",
    "negativeCount": "critic_negative_count",
    "neutralCount": "critic_mixed_count",
    "reviewCount": "critic_total_count",
}

# 3. Same as (2), but for "userReviewSummary/..." variable group
user_fields = {
    "positive": "user_positive_count",
    "negative": "user_negative_count",
    "neutral": "user_mixed_count",
    "reviewCount": "user_total_count",
}

# --- Preparation ---

# 4. Get indices from the "platformReviews/<index>/..." variable group, and sort
#    them into a integer list
indices = sorted(
    int(match.group(1))
    for column in df.columns
    if (match := re.fullmatch(r"platformReviews/(\d+)/name", str(column)))
)

# 5. Some input validation
# Are the column names unique?
if not df.columns.is_unique:
    raise ValueError("The input DataFrame must have unique column names.")
# Are there variables from "platformReview/<index>/.."group?
if not indices:
    raise ValueError("No platformReviews/<index>/name columns were found.")

# 6. Treat pandas nulls and empty strings as missing, while preserving zero.
def is_missing(value):
    return pd.isna(value) or (
        isinstance(value, str) and not value.strip()
    )

# 7. Read review measures as numbers.
#    Unavailable or nonnumeric values, such as 'tbd', become missing.
def numeric(value):
    if is_missing(value):
        return pd.NA
    return pd.to_numeric(value, errors="coerce")

# 8. Define the main output columns, including provenance for critic data.
base_columns = [
    *metadata.values(),       # all new meta-data variable names
    "platform",               # the console the current row corresponds to
    *critic_fields.values(),  # all new critic_fields variable names
    "userscore",              # user score for the console iteration of the game
    *user_fields.values(),    # all new user_fields variable names
    "critic_sourceVar",       # var group supplying critic info
    "critic_review_url",      # console critic review web page
    "userscore_sourceVar",    # source variable for user score
    "user_review_sourceVar",  # source variable for user counts
]

In [13]:
output_rows = []
repeated_critic_rows = []

#test_df = df.sample(n = 10, random_state = 42)
#test_df = df.iloc[:9,:]
test_df = df.copy()

# 9. Process original rows sequentially
for _, game in test_df.iterrows():

    # 10. Build a dictionary containing the platform and the corresponding index
    #     as found in the "platforReviews/.." var group
    platform_slots = {}
    for index in indices:
        platform = game[f"platformReviews/{index}/name"]

        if is_missing(platform):
            continue
        # if the platform key does not exist yet, create it together with an empty list as value.
        # append the index to that list
        platform_slots.setdefault(platform, []).append(index)
    
    # 11. Create one output row for each platform found in this game.
    for platform, slots in platform_slots.items():

        # 12. Create one output row for each platform found in the data for the game        
        record = dict.fromkeys(base_columns, pd.NA) # creates dict with all base columns filled with NAs

        # fills new data rows up with their respective content.
        # repeats game's metadata on this platform row
        for source, destination in metadata.items():
            record[destination] = game.get(source, pd.NA)

        record['platform'] = platform

        # 13. Use the one indexed observation for main critic fields that shows the highest 
        #     critic review count. This ensures the most valid entry to enter the main data

        # get total critic-review counts for this platform's indexed entry
        review_counts = {
            index: numeric(game.get(f"platformReviews/{index}/reviewCount", pd.NA)) for index in slots
        }

        # Select the entry with the highest count. Missing counts rank < 0.
        # Ties retain first indexed platform entry
        selected_index = max(slots, 
                             key = lambda index: (
                                float('-inf')
                                if is_missing(review_counts[index])
                                else review_counts[index])
        )

        selected_prefix = f'platformReviews/{selected_index}'

        for source, destination in critic_fields.items():
            record[destination] = numeric(
                game.get(f"{selected_prefix}/{source}", pd.NA)
            )

        # retain source group and review url
        record[f"critic_sourceVar"] = selected_prefix
        record[f"critic_review_url"] = game.get(
            f"{selected_prefix}/url", pd.NA
        )

        # Indicate whether additional indexed observations exist.
        record["critic_observation_count"] = len(slots)

        # 14. Preserve every observation for repeated platforms in a separate table.
        #     Include all observations, including the one selected for the main table.
        if len(slots) > 1:
            for index in slots:
                prefix = f"platformReviews/{index}"

                observation = {
                    "title": game.get("title", pd.NA),
                    "url": game.get("url", pd.NA),
                    "platform": platform,
                    "source_index": index,
                    "selected_in_main": index == selected_index,
                    "critic_sourceVar": prefix,
                    "critic_review_url": game.get(
                        f"{prefix}/url", pd.NA
                    ),
                }

                # Keep each observation's scores and counts together.
                for source, destination in critic_fields.items():
                    observation[destination] = numeric(
                        game.get(f"{prefix}/{source}", pd.NA)
                    )

                repeated_critic_rows.append(observation)

        # 15. Assign user information to the original section platform only.
        section = game.get("section", pd.NA)

        if not is_missing(section) and platform == section:
            record["userscore"] = numeric(
                game.get("userscore", pd.NA)
            )
            record["userscore_sourceVar"] = "userscore"

            # Apply our provisional section-platform attribution.
            for source, destination in user_fields.items():
                record[destination] = numeric(
                    game.get(f"userReviewsSummary/{source}", pd.NA)
                )

            record["user_review_sourceVar"] = "userReviewsSummary"

        # 16. Retain the platform row even if all review metrics are missing.
        output_rows.append(record)

In [14]:
# 17. build main table and retain the converted nullable dtypes
df_reformatted = pd.DataFrame(output_rows, columns = base_columns + ['critic_observation_count']
                             ).convert_dtypes()

# 18. Define the review table's columns and build separate table of repeated
#     critic observations
review_columns = [
    "title",
    "url",
    "platform",
    "source_index",
    "selected_in_main",
    "critic_sourceVar",
    "critic_review_url",
    *critic_fields.values(),
]

df_critic_repeats = pd.DataFrame(
    repeated_critic_rows,
    columns=review_columns,
).convert_dtypes()

# Preview both tables.
display(df_reformatted.head())
display(df_critic_repeats.head(10))

,title,genre,summary,publisherName,publisherUrl,url,original_platform,ReleaseDate,platform,metascore,...,userscore,user_positive_count,user_negative_count,user_mixed_count,user_total_count,critic_sourceVar,critic_review_url,userscore_sourceVar,user_review_sourceVar,critic_observation_count
0,Tekken 3,3D Fighting,"An ancient evil force has reawakened, attackin...",Namco,https://www.metacritic.com/company/namco/,https://www.metacritic.com/game/tekken-3,PlayStation,1998-04-29,PlayStation,96.0,...,8.9,990.0,35.0,77.0,1102.0,platformReviews/0,https://www.metacritic.com/game/tekken-3/criti...,userscore,userReviewsSummary,1
1,Tekken 3,3D Fighting,"An ancient evil force has reawakened, attackin...",Namco,https://www.metacritic.com/company/namco/,https://www.metacritic.com/game/tekken-3,PlayStation,1998-04-29,Dreamcast,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,platformReviews/1,https://www.metacritic.com/game/tekken-3/criti...,<NA>,<NA>,1
2,Mass Effect 2,Western RPG,The Mass Effect trilogy is a science fiction a...,Electronic Arts,https://www.metacritic.com/company/electronic-...,https://www.metacritic.com/game/mass-effect-2,Xbox 360,2010-01-26,PC,94.0,...,<NA>,<NA>,<NA>,<NA>,<NA>,platformReviews/0,https://www.metacritic.com/game/mass-effect-2/...,<NA>,<NA>,1
3,Mass Effect 2,Western RPG,The Mass Effect trilogy is a science fiction a...,Electronic Arts,https://www.metacritic.com/company/electronic-...,https://www.metacritic.com/game/mass-effect-2,Xbox 360,2010-01-26,Xbox 360,96.0,...,8.9,2792.0,128.0,164.0,3084.0,platformReviews/1,https://www.metacritic.com/game/mass-effect-2/...,userscore,userReviewsSummary,1
4,Mass Effect 2,Western RPG,The Mass Effect trilogy is a science fiction a...,Electronic Arts,https://www.metacritic.com/company/electronic-...,https://www.metacritic.com/game/mass-effect-2,Xbox 360,2010-01-26,PlayStation 3,94.0,...,<NA>,<NA>,<NA>,<NA>,<NA>,platformReviews/2,https://www.metacritic.com/game/mass-effect-2/...,<NA>,<NA>,1


,title,url,platform,source_index,selected_in_main,critic_sourceVar,critic_review_url,metascore,critic_normalized_score,critic_positive_count,critic_negative_count,critic_mixed_count,critic_total_count
0,Major League Baseball 2K5,https://www.metacritic.com/game/major-league-b...,PlayStation 2,0,True,platformReviews/0,https://www.metacritic.com/game/major-league-b...,82.0,81.7838,17.0,0.0,2.0,19.0
1,Major League Baseball 2K5,https://www.metacritic.com/game/major-league-b...,PlayStation 2,3,False,platformReviews/3,https://www.metacritic.com/game/major-league-b...,<NA>,<NA>,3.0,0.0,0.0,3.0
2,Major League Baseball 2K5,https://www.metacritic.com/game/major-league-b...,Xbox,1,True,platformReviews/1,https://www.metacritic.com/game/major-league-b...,81.0,81.2273,21.0,0.0,4.0,25.0
3,Major League Baseball 2K5,https://www.metacritic.com/game/major-league-b...,Xbox,2,False,platformReviews/2,https://www.metacritic.com/game/major-league-b...,88.0,87.75,4.0,0.0,0.0,4.0
4,Uno,https://www.metacritic.com/game/uno,PC,0,True,platformReviews/0,https://www.metacritic.com/game/uno/critic-rev...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
5,Uno,https://www.metacritic.com/game/uno,PC,8,False,platformReviews/8,https://www.metacritic.com/game/uno/critic-rev...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
6,Osmos,https://www.metacritic.com/game/osmos,iOS (iPhone/iPad),1,False,platformReviews/1,https://www.metacritic.com/game/osmos/critic-r...,90.0,89.5455,6.0,0.0,0.0,6.0
7,Osmos,https://www.metacritic.com/game/osmos,iOS (iPhone/iPad),2,True,platformReviews/2,https://www.metacritic.com/game/osmos/critic-r...,88.0,87.8125,8.0,0.0,0.0,8.0
8,Zuma's Revenge!,https://www.metacritic.com/game/zumas-revenge,DS,1,False,platformReviews/1,https://www.metacritic.com/game/zumas-revenge/...,<NA>,<NA>,1.0,0.0,1.0,2.0
9,Zuma's Revenge!,https://www.metacritic.com/game/zumas-revenge,DS,5,True,platformReviews/5,https://www.metacritic.com/game/zumas-revenge/...,75.0,75.125,3.0,0.0,3.0,6.0


**Notes on the output of the reformatting procedure**
- `critic_observation_count` shows how many entries for the same platform were found within the game's original `platformReviews/...` columns.
    - Example: If it says `2`, it means that the corresponding platform appeared twice with separate score and count values (could have been for `platformReviews/0/...` and `platformReviews/3/...`, for example)
    - In case it was > 1, then all appearances for the game are preserved in `df_critic_repeats` for later review

In [15]:
# +++ Validate score agreement +++

# Extract the retained critic score for each game's original platform.
section_scores = df_reformatted.loc[
    df_reformatted["platform"].eq(
        df_reformatted["original_platform"]
    ),
    ["url", "metascore"],
].rename(columns={"metascore": "selected_platform_metascore"})

# Match original scores to their retained platform scores by game URL.
score_check = df[[
        "url",
        "title",
        "metascore",
        "userscore",
        "userReviewsSummary/score",
    ]].merge(
        section_scores,
        on="url",
        how="left",
        validate="one_to_one",
        indicator="section_match",
)

# Compare numbers with a tolerance for float32 rounding.
# Two missing values count as agreement; one missing value does not.
def scores_agree(left, right):
    left = pd.to_numeric(left, errors="coerce").to_numpy(
        dtype=float, na_value=np.nan
    )
    right = pd.to_numeric(right, errors="coerce").to_numpy(
        dtype=float, na_value=np.nan
    )

    return np.isclose(
        left, right, rtol=0, atol=1e-6, equal_nan=True
    )

# Compare the original metascore with the main platform metascore.
score_check["metascore_agrees"] = scores_agree(
    score_check["metascore"],
    score_check["selected_platform_metascore"],
)

# Compare the original userscore with the user-summary score.
score_check["userscore_agrees"] = scores_agree(
    score_check["userscore"],
    score_check["userReviewsSummary/score"],
)

# Retain score discrepancies and missing section-platform matches.
score_exceptions = score_check.loc[
    score_check["section_match"].ne("both")
    | ~score_check["metascore_agrees"]
    | ~score_check["userscore_agrees"]
].copy()

print(f"Score exceptions: {len(score_exceptions):,}")
display(score_exceptions)

Score exceptions: 0


,url,title,metascore,userscore,userReviewsSummary/score,selected_platform_metascore,section_match,metascore_agrees,userscore_agrees


In [25]:
print('Reformatted Metacritic Scraper Data Set: ')
print(f"\n  🔢 Shape: {df_reformatted.shape[0]:,} rows × {df_reformatted.shape[1]} columns")
print(f"  🔄 Duplicates: {df_reformatted.duplicated().sum():,} ({df_reformatted.duplicated().sum()/len(df_reformatted)*100:.2f}%)")
print(f"  💾 Memory Usage: {df_reformatted.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print('\nData set containing games with critic repeats for the same platform: ')
print(f"\n  🔢 Shape: {df_critic_repeats.shape[0]:,} rows × {df_critic_repeats.shape[1]} columns")
repeats_games = len(df_critic_repeats['title'].unique())
repeats_games_pc = np.round(repeats_games / len(df_reformatted['title'].unique()) * 100, 2)
print(f'  Number of games containing repeats: {repeats_games} ({repeats_games_pc}% of games)')

Reformatted Metacritic Scraper Data Set: 

  🔢 Shape: 35,162 rows × 25 columns
  🔄 Duplicates: 0 (0.00%)
  💾 Memory Usage: 57.08 MB

Data set containing games with critic repeats for the same platform: 

  🔢 Shape: 61 rows × 13 columns
  Number of games containing repeats: 26 (0.19% of games)


In [ ]:
# Save the results

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
df_reformatted.to_csv(DATA_FORMATTED_PATH, index=False)
df_critic_repeats.to_csv(PROCESSED_DATA_DIR / 'critic_platform_repeats.csv', index=False)
